In [13]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdMolDescriptors
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

RDLogger.DisableLog("rdApp.*")


# =========================
# User settings
# =========================
ROOT = Path("/home/sy/SY/N-体系")
EXCEL_FILE = ROOT / "FE clac.xlsx"
SHEET_NAME = "total"

COMMERCIAL_LIB = ROOT / "commercial_n_library" / "mcule_instock_n_containing_organic_molecules.csv"

OUT_DIR = ROOT / "ML" / "outputs" / "commercial_plus_co2rr_n_map_revised"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_ALL = OUT_DIR / "combined_tsne_coordinates.csv"
OUT_PLOT = OUT_DIR / "commercial_plus_co2rr_n_map_revised.png"
OUT_CLUSTER_SUMMARY = OUT_DIR / "cluster_feature_summary.csv"
OUT_CLUSTER_EXAMPLES = OUT_DIR / "cluster_representative_examples.csv"
OUT_CLUSTER_TEXT = OUT_DIR / "cluster_feature_summary_readable.txt"
OUT_COMMERCIAL = OUT_DIR / "commercial_library_tsne_coordinates.csv"
OUT_MODIFIERS = OUT_DIR / "n_modifiers_tsne_coordinates.csv"

FP_RADIUS = 2
FP_NBITS = 256
N_CLUSTERS = 3
TSNE_RANDOM_STATE = 42
KMEANS_RANDOM_STATE = 87
MAX_COMMERCIAL = 5000
FG_JITTER = 0.75
ROUND_COLUMNS = ["Round 1", "Round 2", "Round 3", "Round 4", "Round 5"]
ROUND_COLORS = {
    "Initial dataset": "#F6D746",  # yellow
    "Round 1": "#E67E22",          # orange
    "Round 2": "#E56AA6",          # pink
    "Round 3": "#D73027",          # red
}

ROUND_MARKERS = {
    "Initial dataset": "^",  # triangle
    "Round 1": "o",          # circle
    "Round 2": "s",          # square
    "Round 3": "D",          # diamond
}


FEATURE_SMARTS = {
    "aromatic_n_heterocycle": "[n]",
    "pyridine_like_n": "[nH0;r6]",
    "pyrrolic_n": "[nH]",
    "aliphatic_amine": "[NX3;H2,H1;!$(NC=O)]",
    "tertiary_amine": "[NX3;H0;!$(NC=O)]",
    "amide_or_urea": "[NX3][CX3](=[OX1])",
    "nitrile": "[CX2]#N",
    "azo_or_imine_n": "[NX2]=[CX3,NX2]",
    "quaternary_ammonium": "[NX4+]",
    "aniline_like": "[NX3;H2,H1]c",
}

FEATURE_PATTERNS = {name: Chem.MolFromSmarts(smarts) for name, smarts in FEATURE_SMARTS.items()}

FEATURE_LABELS = {
    "aromatic_n_heterocycle": "aromatic N-heterocycles",
    "pyridine_like_n": "pyridine-like N",
    "pyrrolic_n": "pyrrolic N",
    "aliphatic_amine": "aliphatic amines",
    "tertiary_amine": "tertiary amines",
    "amide_or_urea": "amide/urea-type N",
    "nitrile": "nitrile groups",
    "azo_or_imine_n": "imine/azo-like N",
    "quaternary_ammonium": "quaternary ammonium",
    "aniline_like": "aniline-like N",
    "is_aromatic": "overall aromatic character",
    "n_atom_count": "number of N atoms",
    "ring_count": "ring count",
    "aromatic_ring_count": "aromatic ring count",
}


def mol_from_smiles(smiles: str):
    mol = Chem.MolFromSmiles(str(smiles).strip())
    if mol is None:
        return None
    atom_symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
    if "N" not in atom_symbols:
        return None
    if "C" not in atom_symbols:
        return None
    return mol


def fp_array_from_mol(mol) -> np.ndarray:
    fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(
        mol,
        radius=FP_RADIUS,
        nBits=FP_NBITS,
    )
    arr = np.zeros((FP_NBITS,), dtype=float)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def detect_feature_flags(mol) -> dict:
    flags = {}
    for name, pattern in FEATURE_PATTERNS.items():
        flags[name] = int(pattern is not None and mol.HasSubstructMatch(pattern))
    flags["is_aromatic"] = int(any(atom.GetIsAromatic() for atom in mol.GetAtoms()))
    flags["n_atom_count"] = sum(1 for atom in mol.GetAtoms() if atom.GetSymbol() == "N")
    flags["ring_count"] = rdMolDescriptors.CalcNumRings(mol)
    flags["aromatic_ring_count"] = rdMolDescriptors.CalcNumAromaticRings(mol)
    return flags


def infer_round_label(row: pd.Series, round_cols_present: list[str]) -> str:
    for col in round_cols_present:
        try:
            value = float(row[col])
        except Exception:
            value = 0.0
        if value == 1:
            if col in {"Round 1", "Round 2"}:
                return "Initial dataset"
            if col == "Round 3":
                return "Round 1"
            if col == "Round 4":
                return "Round 2"
            if col == "Round 5":
                return "Round 3"
            return col
    return "Unassigned"


def load_commercial_library() -> pd.DataFrame:
    df = pd.read_csv(COMMERCIAL_LIB)
    df = df.drop_duplicates(subset=["smiles"]).copy()
    if len(df) > MAX_COMMERCIAL:
        df = df.sample(MAX_COMMERCIAL, random_state=TSNE_RANDOM_STATE).reset_index(drop=True)

    records = []
    for _, row in df.iterrows():
        smiles = str(row["smiles"]).strip()
        mol = mol_from_smiles(smiles)
        if mol is None:
            continue
        records.append(
            {
                "SMILES": smiles,
                "Source_Label": "Commercial library",
                "Fingerprint_Product": fp_array_from_mol(mol),
                **detect_feature_flags(mol),
            }
        )
    return pd.DataFrame(records)


def load_co2rr_molecules() -> pd.DataFrame:
    df = pd.read_excel(EXCEL_FILE, sheet_name=SHEET_NAME)
    df = df.loc[:, [c for c in df.columns if pd.notna(c)]].copy()
    df[["No", "CAS No.", "SMILES"]] = df[["No", "CAS No.", "SMILES"]].ffill()
    round_cols_present = [col for col in ROUND_COLUMNS if col in df.columns]
    if round_cols_present:
        df[round_cols_present] = df[round_cols_present].ffill().fillna(0)
    raw_rows = len(df)
    keep_cols = ["No", "CAS No.", "SMILES"] + round_cols_present
    df = df[keep_cols].dropna(subset=["SMILES"]).copy()
    df["No"] = df["No"].astype(str).str.strip()
    df["SMILES"] = df["SMILES"].astype(str).str.strip()
    df["CAS No."] = df["CAS No."].astype(str).str.strip()
    before_dedup = len(df)
    df = df.drop_duplicates(subset=["No"], keep="first").reset_index(drop=True)
    print(
        f"    CO2RR table rows after dropping empty SMILES: {before_dedup} "
        f"(raw rows: {raw_rows}; unique molecules by No.: {len(df)})"
    )

    records = []
    invalid_smiles = 0
    filtered_non_nc = 0
    for _, row in df.iterrows():
        smiles = str(row["SMILES"]).strip()
        mol = mol_from_smiles(smiles)
        if mol is None:
            parsed = Chem.MolFromSmiles(smiles)
            if parsed is None:
                invalid_smiles += 1
            else:
                filtered_non_nc += 1
            continue
        records.append(
            {
                "No": row["No"],
                "CAS No.": row["CAS No."],
                "SMILES": smiles,
                "Source_Label": "N-modifiers",
                "Active_Learning_Round": infer_round_label(row, round_cols_present),
                "Fingerprint_Product": fp_array_from_mol(mol),
                **detect_feature_flags(mol),
            }
        )
    print(
        f"    Kept valid N-containing organic molecules: {len(records)} "
        f"(invalid SMILES: {invalid_smiles}; filtered as non N/C molecules: {filtered_non_nc})"
    )
    return pd.DataFrame(records)


def build_combined_map(commercial_df: pd.DataFrame, modifiers_df: pd.DataFrame):
    combined_df = pd.concat([commercial_df, modifiers_df], ignore_index=True, sort=False)
    fingerprints = np.vstack(combined_df["Fingerprint_Product"].to_numpy())

    kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=KMEANS_RANDOM_STATE, n_init=20)
    raw_cluster = kmeans.fit_predict(fingerprints)

    combined_df = combined_df.copy()
    combined_df["_raw_cluster"] = raw_cluster
    combined_df["_distance_to_center"] = np.linalg.norm(
        fingerprints - kmeans.cluster_centers_[raw_cluster],
        axis=1,
    )

    cluster_sizes = combined_df.groupby("_raw_cluster").size().sort_values(ascending=False).index.tolist()
    remap = {cluster_id: idx for idx, cluster_id in enumerate(cluster_sizes)}
    combined_df["Cluster"] = combined_df["_raw_cluster"].map(remap)

    tsne = TSNE(
        n_components=2,
        random_state=TSNE_RANDOM_STATE,
        init="pca",
        learning_rate=180,
        perplexity=45,
    )
    reduced = tsne.fit_transform(fingerprints)
    combined_df["Dimension_1"] = reduced[:, 0]
    combined_df["Dimension_2"] = reduced[:, 1]
    combined_df = combined_df.drop(columns=["_raw_cluster"])

    commercial_mapped_df = combined_df[combined_df["Source_Label"] == "Commercial library"].copy()
    modifiers_mapped_df = combined_df[combined_df["Source_Label"] == "N-modifiers"].copy()
    return commercial_mapped_df, modifiers_mapped_df, combined_df


def draw_map(commercial_df: pd.DataFrame, modifiers_df: pd.DataFrame):
    colors = ["#2CA02C", "#3B6FF5", "#8E24AA"]

    plt.rcParams.update(
        {
            "font.family": "Liberation Sans",
            "font.size": 18,
            "font.weight": "bold",
            "axes.labelsize": 18,
            "axes.labelweight": "bold",
            "axes.titlesize": 18,
            "axes.titleweight": "bold",
            "xtick.labelsize": 18,
            "ytick.labelsize": 18,
        }
    )

    fig, ax = plt.subplots(figsize=(8.0, 6.6), dpi=600)

    for cluster in range(N_CLUSTERS):
        mask = commercial_df["Cluster"] == cluster
        ax.scatter(
            commercial_df.loc[mask, "Dimension_1"],
            commercial_df.loc[mask, "Dimension_2"],
            c=colors[cluster],
            label=f"Class {cluster + 1}",
            s=18,
            alpha=0.88,
            edgecolors="none",
            zorder=1,
        )

    jitter_rng = np.random.default_rng(20260613)
    fg_df = modifiers_df.copy()
    fg_df["_plot_x"] = fg_df["Dimension_1"].to_numpy() + jitter_rng.normal(0, FG_JITTER, len(modifiers_df))
    fg_df["_plot_y"] = fg_df["Dimension_2"].to_numpy() + jitter_rng.normal(0, FG_JITTER, len(modifiers_df))

    plot_order = ["Initial dataset", "Round 1", "Round 2", "Round 3"]
    plotted_rounds = [label for label in plot_order if label in fg_df["Active_Learning_Round"].astype(str).unique()]
    extra_rounds = [
        label for label in fg_df["Active_Learning_Round"].dropna().astype(str).unique()
        if label not in plotted_rounds
    ]
    for round_label in plotted_rounds + extra_rounds:
        sub = fg_df[fg_df["Active_Learning_Round"] == round_label].copy()
        if sub.empty:
            continue
        ax.scatter(
            sub["_plot_x"],
            sub["_plot_y"],
            c=ROUND_COLORS.get(round_label, "#F6D746"),
            marker=ROUND_MARKERS.get(round_label, "o"),
            s=30,
            alpha=1.0,
            edgecolors="white",
            linewidths=0.7,
            label=round_label if round_label != "Unassigned" else "N-modifiers tested",
            zorder=6,
        )

    for spine in ["top", "bottom", "left", "right"]:
        ax.spines[spine].set_linewidth(1.5)

    all_x = pd.concat([commercial_df["Dimension_1"], modifiers_df["Dimension_1"]], ignore_index=True)
    all_y = pd.concat([commercial_df["Dimension_2"], modifiers_df["Dimension_2"]], ignore_index=True)
    x_pad = max((all_x.max() - all_x.min()) * 0.18, 9.0)
    y_pad = max((all_y.max() - all_y.min()) * 0.2, 9.0)
    ax.set_xlim(all_x.min() - x_pad, all_x.max() + x_pad)
    ax.set_ylim(all_y.min() - y_pad, all_y.max() + y_pad)

    ax.tick_params(width=1.4, length=5)
    for label in ax.get_xticklabels():
        label.set_fontweight("bold")
    for label in ax.get_yticklabels():
        label.set_fontweight("bold")

    ax.set_xlabel("t-SNE Dimension 1")
    ax.set_ylabel("t-SNE Dimension 2")
    ax.set_title("t-SNE Map of Commercial N Library and N-modifiers", pad=10)
    ax.legend(
        frameon=False,
        fontsize=13,
        loc="upper right",
        bbox_to_anchor=(0.985, 0.99),
        ncol=4,
        handletextpad=0.7,
        columnspacing=1.4,
        borderaxespad=0.2,
    )

    plt.tight_layout()
    plt.savefig(OUT_PLOT, dpi=800, bbox_inches="tight")
    plt.close()


def summarize_clusters(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    feature_cols = list(FEATURE_SMARTS.keys()) + ["is_aromatic"]
    numeric_cols = ["n_atom_count", "ring_count", "aromatic_ring_count"]

    summary_rows = []
    example_rows = []
    for cluster_id in sorted(df["Cluster"].unique()):
        sub = df[df["Cluster"] == cluster_id].copy()
        row = {
            "cluster": int(cluster_id) + 1,
            "n_molecules": len(sub),
            "n_n_modifiers": int((sub["Source_Label"] == "N-modifiers").sum()),
            "n_commercial_library": int((sub["Source_Label"] == "Commercial library").sum()),
        }
        for col in feature_cols:
            row[f"{col}_fraction"] = float(sub[col].mean())
        for col in numeric_cols:
            row[f"{col}_mean"] = float(sub[col].mean())

        feature_rank = sorted(
            [(col, float(sub[col].mean())) for col in feature_cols],
            key=lambda x: x[1],
            reverse=True,
        )
        row["top_feature_tags"] = " | ".join(
            f"{FEATURE_LABELS[name]} ({value:.2f})" for name, value in feature_rank[:4]
        )
        summary_rows.append(row)

        reps = sub.nsmallest(8, "_distance_to_center")
        for _, rep in reps.iterrows():
            example_rows.append(
                {
                    "cluster": int(cluster_id) + 1,
                    "source_label": rep["Source_Label"],
                    "No": rep.get("No", np.nan),
                    "CAS No.": rep.get("CAS No.", np.nan),
                    "SMILES": rep["SMILES"],
                    "distance_to_center": rep["_distance_to_center"],
                }
            )

    return pd.DataFrame(summary_rows), pd.DataFrame(example_rows)


def build_cluster_readable_summary(cluster_summary_df: pd.DataFrame) -> str:
    text_blocks = []
    feature_cols = list(FEATURE_SMARTS.keys()) + ["is_aromatic"]
    numeric_cols = ["n_atom_count", "ring_count", "aromatic_ring_count"]

    for _, row in cluster_summary_df.sort_values("cluster").iterrows():
        ranked_features = sorted(
            [(col, float(row[f"{col}_fraction"])) for col in feature_cols],
            key=lambda x: x[1],
            reverse=True,
        )[:4]
        feature_part = "; ".join(
            f"{FEATURE_LABELS[col]} ({value:.2f})" for col, value in ranked_features
        )
        numeric_part = ", ".join(
            f"{FEATURE_LABELS[col]} = {float(row[f'{col}_mean']):.2f}" for col in numeric_cols
        )
        text_blocks.append(
            f"Class {int(row['cluster'])}: mainly enriched in {feature_part}. "
            f"Average structural features: {numeric_part}."
        )

    return "\n".join(text_blocks)


def main():
    print("[1] Loading commercial N-containing library")
    commercial_df = load_commercial_library()
    print(f"    Commercial molecules used: {len(commercial_df)}")

    print("[2] Loading CO2RR modifier molecules")
    modifiers_df = load_co2rr_molecules()
    print(f"    N-modifiers used: {len(modifiers_df)}")

    print("[3] Building combined chemical space from commercial library + N-modifiers")
    commercial_mapped_df, modifiers_mapped_df, combined_df = build_combined_map(commercial_df, modifiers_df)

    print("[4] Saving coordinate tables")
    save_combined_df = combined_df.copy()
    save_combined_df["Fingerprint_Product"] = save_combined_df["Fingerprint_Product"].apply(
        lambda x: ",".join(map(str, x.astype(int)))
    )
    save_combined_df.to_csv(OUT_ALL, index=False)

    save_commercial_df = commercial_mapped_df.copy()
    save_commercial_df["Fingerprint_Product"] = save_commercial_df["Fingerprint_Product"].apply(
        lambda x: ",".join(map(str, x.astype(int)))
    )
    save_commercial_df.to_csv(OUT_COMMERCIAL, index=False)

    save_modifiers_df = modifiers_mapped_df.copy()
    save_modifiers_df["Fingerprint_Product"] = save_modifiers_df["Fingerprint_Product"].apply(
        lambda x: ",".join(map(str, x.astype(int)))
    )
    save_modifiers_df.to_csv(OUT_MODIFIERS, index=False)

    print("[5] Summarizing cluster features")
    cluster_summary_df, cluster_examples_df = summarize_clusters(combined_df)
    cluster_summary_df.to_csv(OUT_CLUSTER_SUMMARY, index=False)
    cluster_examples_df.to_csv(OUT_CLUSTER_EXAMPLES, index=False)
    OUT_CLUSTER_TEXT.write_text(build_cluster_readable_summary(cluster_summary_df), encoding="utf-8")

    print("[6] Drawing map")
    print(
        "    N-modifier plotted points: "
        f"{len(modifiers_mapped_df)} total, "
        f"{modifiers_mapped_df[['Dimension_1', 'Dimension_2']].round(3).drop_duplicates().shape[0]} unique rounded positions"
    )
    draw_map(commercial_mapped_df, modifiers_mapped_df)

    print(f"[Done] Combined coordinates saved to: {OUT_ALL}")
    print(f"[Done] Commercial-library coordinates saved to: {OUT_COMMERCIAL}")
    print(f"[Done] N-modifier coordinates saved to: {OUT_MODIFIERS}")
    print(f"[Done] Cluster feature summary saved to: {OUT_CLUSTER_SUMMARY}")
    print(f"[Done] Cluster representative examples saved to: {OUT_CLUSTER_EXAMPLES}")
    print(f"[Done] Cluster readable summary saved to: {OUT_CLUSTER_TEXT}")
    print(f"[Done] Plot saved to: {OUT_PLOT}")


if __name__ == "__main__":
    main()


[1] Loading commercial N-containing library
    Commercial molecules used: 5000
[2] Loading CO2RR modifier molecules
    CO2RR table rows after dropping empty SMILES: 666 (raw rows: 666; unique molecules by No.: 111)
    Kept valid N-containing organic molecules: 110 (invalid SMILES: 1; filtered as non N/C molecules: 0)
    N-modifiers used: 110
[3] Building combined chemical space from commercial library + N-modifiers
[4] Saving coordinate tables
[5] Summarizing cluster features
[6] Drawing map
    N-modifier plotted points: 110 total, 109 unique rounded positions
[Done] Combined coordinates saved to: /home/sy/SY/N-体系/ML/outputs/commercial_plus_co2rr_n_map_revised/combined_tsne_coordinates.csv
[Done] Commercial-library coordinates saved to: /home/sy/SY/N-体系/ML/outputs/commercial_plus_co2rr_n_map_revised/commercial_library_tsne_coordinates.csv
[Done] N-modifier coordinates saved to: /home/sy/SY/N-体系/ML/outputs/commercial_plus_co2rr_n_map_revised/n_modifiers_tsne_coordinates.csv
[Done] 